# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My method: I'm using Logistic Regression as my first model, then Random Forest as a stronger comparison. My lane is a yes/no prediction (declining vs. not) used to rank pages by review priority — this fits the "yes/no with an observed label, evaluated as a ranking" pattern. I start simple (logistic regression is fully readable — you can see exactly which features push the score up or down) and only add complexity (random forest) if it earns its keep by measurably beating the simpler model.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a grouped split by client_id, ensuring every client's pages fall entirely into either training or testing — never split between both. This matters because a model tested on clients it already saw during training could "memorize" client-specific quirks instead of learning general patterns, producing an inflated, dishonest score.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Load your data fresh
df = pd.read_csv('FlyRank-Internship/data/raw/content_refresh_anonymized.csv')

# Build the label: this is what we're predicting (never use trend_direction as a FEATURE)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Grouped split: every client's pages stay entirely in train OR entirely in test
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")

# Confirm zero overlap — this IS your proof the split is honest
overlap = set(train_df['client_id']) & set(test_df['client_id'])
print("Clients appearing in both sets (should be 0):", len(overlap))

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compared random guessing, my Week-4 baseline rule, logistic regression, and random forest — all on the exact same held-out test clients, using precision@20 and precision@50. [Fill in: state your actual printed numbers here, and which method won.]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# --- Features: everything EXCEPT the label and ID/leakage columns ---
feature_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr',
                 'engagement_rate', 'scroll_rate', 'content_age_days',
                 'days_since_last_update', 'word_count']

X_train = train_df[feature_cols].fillna(0)
y_train = train_df['is_declining_label']
X_test = test_df[feature_cols].fillna(0)
y_test = test_df['is_declining_label']

# --- Train Logistic Regression ---
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
test_df['log_reg_score'] = log_reg.predict_proba(X_test)[:, 1]

# --- Train Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
test_df['rf_score'] = rf.predict_proba(X_test)[:, 1]

# --- Recompute your Week-4 baseline score ON THIS SAME TEST SET (fair comparison) ---
test_df['visibility_score'] = np.log1p(test_df['impressions_90d']).rank(pct=True)
test_df['freshness_risk_score'] = test_df['days_since_last_update'].rank(pct=True)
test_df['position_opportunity_score'] = (
    (1 - test_df['avg_position'].clip(lower=1, upper=50).rank(pct=True))
    * test_df['visibility_score'] * (test_df['avg_position'] > 0).astype(int)
)
test_df['depth_gap_score'] = (1 - test_df['word_count'].rank(pct=True)) * test_df['visibility_score']
test_df['baseline_score'] = (
    0.40 * test_df['visibility_score'] + 0.30 * test_df['freshness_risk_score']
    + 0.25 * test_df['position_opportunity_score'] + 0.05 * test_df['depth_gap_score']
).clip(0, 1)

# --- Precision@K function ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Build the comparison table ---
base_rate = y_test.mean()
comparison = pd.DataFrame({
    'Method': ['Random guessing (base rate)', 'Baseline rule', 'Logistic Regression', 'Random Forest'],
    'Precision@20': [
        base_rate,
        precision_at_k(test_df['baseline_score'], y_test, 20),
        precision_at_k(test_df['log_reg_score'], y_test, 20),
        precision_at_k(test_df['rf_score'], y_test, 20),
    ],
    'Precision@50': [
        base_rate,
        precision_at_k(test_df['baseline_score'], y_test, 50),
        precision_at_k(test_df['log_reg_score'], y_test, 50),
        precision_at_k(test_df['rf_score'], y_test, 50),
    ],
})
print(comparison.round(3))

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model relies most heavily on [name your top 2-3 features from the printed list] — this makes sense because [explain why, in plain words]. Looking at concrete errors, the model's false positives tend to be [describe a pattern you notice, e.g. "pages with high impressions but normal freshness"], while false negatives tend to be [describe another pattern]. This tells me the model is [directional/decision-support — never "proven" or "guaranteed"] at identifying pages that need review, particularly strong at [X], weaker at [Y].

In [ ]:
# --- What does the model lean on? (feature importance) ---
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("Top features the Random Forest relies on:")
print(importances.head(5))

# --- Where is the model most wrong? Look at false positives (predicted declining, actually wasn't) ---
test_df['rf_predicted'] = (test_df['rf_score'] >= 0.5).astype(int)
false_positives = test_df[(test_df['rf_predicted'] == 1) & (test_df['is_declining_label'] == 0)]
false_negatives = test_df[(test_df['rf_predicted'] == 0) & (test_df['is_declining_label'] == 1)]

print(f"\nFalse positives (flagged declining, actually fine): {len(false_positives)}")
print(f"False negatives (missed real declining pages): {len(false_negatives)}")

# Show 3 concrete wrong cases
print("\n3 example false positives:")
print(false_positives[['content_id', 'impressions_90d', 'avg_position', 'days_since_last_update']].head(3))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.